# Task 1 — The Restricted Robot
### Testing: Span & Basis

## Assignment scenario

You are programming a grid-based AI agent. Due to hardware restrictions, its
movement protocol is limited to linear combinations of two specific vectors:

- **Move A**: 2 steps Forward (x), 1 step Right (y)  →  vector `(2, 1)`
- **Move B**: 1 step Forward (x), 2 steps Left (-y)  →  vector `(1, -2)`

**Objective:** build a purely mathematical solver, in raw Python, that
determines whether the robot can reach a specific target coordinate,
given that it can only execute **whole-number (integer) multiples** of
Move A and Move B (in either direction — forward or backward along each
vector).

**Constraint:** no NumPy, no Pandas, no external math/algebra library of
any kind. Every vector, matrix, and operation below is built from plain
Python lists, tuples, and loops (`src/linalg.py`), plus the `fractions`
module for exact (non-floating-point) arithmetic where it matters.

## Step 1 — Setting up

We import our own from-scratch linear algebra library. Every function in
`src/linalg.py` (determinant, rank, Gaussian elimination, extended
Euclidean algorithm) is implemented with plain Python — go take a look.

In [ ]:
import sys
sys.path.insert(0, ".")
from linalg import (
    columns_to_matrix, det, rank, is_linearly_independent, spans_r2,
    egcd, gcd, vec_add, scalar_mult
)
from fractions import Fraction
import matplotlib.pyplot as plt

# The two move vectors given in the assignment
MOVE_A = (2, 1)    # 2 forward, 1 right
MOVE_B = (1, -2)   # 1 forward, 2 left
MOVE_VECTORS = [MOVE_A, MOVE_B]
print("Move A:", MOVE_A)
print("Move B:", MOVE_B)


## Step 2 — Span: what region can the robot reach, ignoring "whole steps" for a moment?

Before worrying about *integer* multiples, ask the simpler continuous
question: if the robot could move by any *real-numbered* multiple of Move A
and Move B, what could it reach?

- If Move A and Move B are linearly independent (neither is a scalar
  multiple of the other), together they **span all of R^2** — the whole
  plane.
- If they were parallel, they'd only span a line.

We check this with **rank**: build a matrix with the move vectors as
columns, and compute its rank via Gaussian elimination (`rref` under the
hood in our library).

In [ ]:
def describe_span(move_vectors):
    A = columns_to_matrix(move_vectors)
    r = rank(A)
    if r == 0:
        return "Span = just the origin. This robot cannot move at all."
    elif r == 1:
        return "Span = a single line through the origin (1-dimensional)."
    else:
        return "Span = all of R^2 (the move vectors are linearly independent)."

print(MOVE_VECTORS, "->", describe_span(MOVE_VECTORS))


## Step 3 — Basis: is the movement set "efficient"?

A set of vectors is a **basis** for R^2 if it (a) spans R^2 and (b) is
linearly independent (no redundant vectors). For exactly 2 vectors in R^2,
this reduces to: are they linearly independent, i.e. is their determinant
**nonzero**?

If yes, neither Move A nor Move B is wasted — each contributes a genuinely
new direction the robot can travel in.

In [ ]:
def is_basis_r2(move_vectors):
    if len(move_vectors) != 2:
        return False
    return is_linearly_independent(move_vectors)

A = columns_to_matrix(MOVE_VECTORS)
d = det(A)
print("Matrix of move vectors:")
for row in A:
    print(row)
print()
print("det =", d)
print("Is {Move A, Move B} a basis for R^2?", is_basis_r2(MOVE_VECTORS))


## Step 4 — The real question: the INTEGER lattice

Being a basis of R^2 is *not* the same as being able to reach every grid
cell. The robot can only take an **integer** number of steps of each move
vector (positive or negative). The set of reachable points is the
**lattice** generated by the move vectors:

```
L = { n1 * MOVE_A + n2 * MOVE_B : n1, n2 are integers }
```

**Key fact:** if `M` is the matrix with the move vectors as columns, the
lattice `L` contains exactly 1 out of every `|det(M)|` grid cells.

- `|det(M)| == 1` → every integer point is reachable.
- `|det(M)| > 1` → only a fraction of the grid is reachable.
- `det(M) == 0` → the reachable set collapses to a line (covered in
  Step 2 already, doesn't apply here since we already found det != 0).

In [ ]:
lattice_index = abs(d)
print("Lattice index |det(M)| =", lattice_index)
if lattice_index == 1:
    print("-> Move A and Move B form a basis for the FULL integer grid Z^2.")
    print("   Every single coordinate on the grid is reachable.")
else:
    print(f"-> Only 1 out of every {lattice_index} grid cells is reachable.")


## Step 5 — The solver: can the robot reach a specific target?

Now the core deliverable: **given a target cell `(tx, ty)`, can the robot
get there using integer multiples of Move A and Move B?**

This is a 2-variable **linear Diophantine system**:

```
n1 * MOVE_A_x + n2 * MOVE_B_x = tx
n1 * MOVE_A_y + n2 * MOVE_B_y = ty
```

We solve for `(n1, n2)` using **Cramer's rule** — built entirely from our
own `det()` function — with exact `Fraction` arithmetic so there is never
any floating-point rounding risk. If both `n1` and `n2` come out as whole
numbers, the target is reachable; if either has a leftover fractional
part, it isn't.

In [ ]:
def can_reach(target, move_vectors=MOVE_VECTORS):
    """Raw-Python solver. Returns (reachable: bool, (n1, n2) or None).
    No NumPy, no Pandas -- just det() from our own linalg.py and Fraction
    for exact arithmetic."""
    v1, v2 = move_vectors
    M = columns_to_matrix([v1, v2])
    D = det(M)

    if D == 0:
        # Degenerate case: move vectors don't span R^2. Not needed for
        # this assignment's move set, but handled for completeness.
        return (target == (0, 0)), None

    # Cramer's rule: replace each column of M with the target vector in turn
    M1 = [[target[0], M[0][1]], [target[1], M[1][1]]]
    M2 = [[M[0][0], target[0]], [M[1][0], target[1]]]
    n1 = Fraction(det(M1), D)
    n2 = Fraction(det(M2), D)
    reachable = (n1.denominator == 1) and (n2.denominator == 1)
    return reachable, (n1, n2)


# --- Solve for a batch of candidate targets ---
targets = [(0, 0), (1, 0), (3, -1), (5, -4), (2, 1), (4, 2), (1, 1), (10, -5)]

print(f"{'target':>10} | {'reachable?':>10} | steps (n1, n2)")
print("-" * 45)
for t in targets:
    reachable, steps = can_reach(t)
    print(f"{str(t):>10} | {str(reachable):>10} | {steps}")


## Step 6 — Sanity-check a specific reachable target by hand

Let's verify one result manually to build confidence in the solver. Take
`(10, -5)`: the solver reported this as reachable, so it should hand back a
whole-number `(n1, n2)` that we can plug back in and reconstruct exactly.

In [ ]:
target = (10, -5)
reachable, (n1, n2) = can_reach(target)
print(f"Target {target}: reachable = {reachable}, n1 = {n1}, n2 = {n2}")

# Manually reconstruct the path to double-check
check = vec_add(scalar_mult(n1, MOVE_A), scalar_mult(n2, MOVE_B))
print(f"{n1} * {MOVE_A} + {n2} * {MOVE_B} = {check}")
print("Matches target?", check == target)

# And for contrast, a target the solver correctly rejects
bad_target = (1, 0)
reachable2, coeffs2 = can_reach(bad_target)
print()
print(f"Target {bad_target}: reachable = {reachable2}, exact (n1, n2) = {coeffs2}  <- not both whole numbers, so unreachable")


## Step 7 — Visualize the reachable region

Turning the abstract determinant math into something you can see: a grid
of points, colored by whether the robot's exact Move A / Move B combo can
reach them.

In [ ]:
def plot_reachable(move_vectors, extent=8, title=None):
    reachable_pts, unreachable_pts = [], []
    for x in range(-extent, extent + 1):
        for y in range(-extent, extent + 1):
            ok, _ = can_reach((x, y), move_vectors)
            (reachable_pts if ok else unreachable_pts).append((x, y))

    fig, ax = plt.subplots(figsize=(5.5, 5.5))
    if unreachable_pts:
        ux, uy = zip(*unreachable_pts)
        ax.scatter(ux, uy, color="lightgray", s=25, label="unreachable")
    if reachable_pts:
        rx, ry = zip(*reachable_pts)
        ax.scatter(rx, ry, color="crimson", s=25, label="reachable")
    ax.scatter([0], [0], color="black", s=70, marker="*", label="start")
    ax.set_aspect("equal")
    ax.set_title(title or f"move vectors = {move_vectors}")
    ax.legend(loc="upper right", fontsize=8)
    ax.grid(True, linewidth=0.3)
    plt.show()

plot_reachable(MOVE_VECTORS, title="Move A = (2,1), Move B = (1,-2)  (assignment scenario)")

# For comparison: a degenerate move set where the robot gets stuck on a
# fraction of the grid (|det| = 2), to make the assignment's basis clearly
# visible as a special, "lucky" case where it isn't stuck at all.
plot_reachable([(2, 0), (0, 2)], title="Contrast: (2,0),(0,2) -- only even cells reachable")


## Step 8 — Exercises for interns

1. **Verify the basis claim visually.** Look at the first plot above. Does
   every visible grid cell show up red (reachable)? Cross-check this
   against the `det` and `lattice_index` values computed in Steps 3–4.
2. **Try a knight-move robot.** Chess-knight-style moves `(1, 2)` and
   `(2, 1)`. Is this a basis for Z^2? Which of `(1, 0)`, `(1, 1)`,
   `(0, 1)` are reachable?
3. **Break the basis on purpose.** Change Move B to `(4, 2)` (a scalar
   multiple of Move A). What happens to `det`, to `describe_span`, and to
   the reachable-cells plot? Explain why in your own words.
4. **Minimum number of moves.** For a reachable target, is `(n1, n2)`
   from Cramer's rule always unique? If so, why? (Hint: this is a direct
   consequence of Move A and Move B being linearly independent — a basis
   gives *unique* coordinates for every point in the span.)